In [1]:
from B0_MODEL import UNET
from B2_UTILS import (load_checkpoint, save_checkpoint, get_loaders,
                      check_accuracy, save_predictions_as_imgs)
import torch
from tqdm import tqdm  # makes your loops show a smart progress meter
import torch.nn as nn
import torch.optim as optim
import os
import json

In [3]:
# Set training epoch munbers
TRAIN = True
training_epoch_num = 1000

# Test training?
BATCH_1_TEST = True


In [21]:
# ====================== Hyperparameters etc. ========================
MAIN_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST'
MODEL_TYPES = ['secondary_strctures', 'key_atoms', 'residue_types']
MAP_DIRS_TYPES = [['map_sample'], ['map_sample', 'secondary_strctures'],
                  ['map_sample', 'secondary_strctures', 'key_atoms']]

type_index = 0
model_type = MODEL_TYPES[type_index]
map_dirs = MAP_DIRS_TYPES[type_index]

print(model_type)

secondary_strctures


In [5]:
# ====================== Hyperparameters etc. ========================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 8  # batch_size vs GPU Memory: 1vs2877; 2vs3485; 4vs4668; 8vs7162

num_workers = 2
learning_rate = 1e-4
pin_memory = True
IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_DEPTH = [64, 64, 64]

# Set input data
data_path = os.path.join(MAIN_PATH, 'train_val_data')
train_map_dirs = [
    os.path.join(data_path, 'train', map_dir) for map_dir in map_dirs
]
val_map_dirs = [
    os.path.join(data_path, 'val', map_dir) for map_dir in map_dirs
]
train_model_dir = os.path.join(data_path, 'train', model_type)
val_model_dir = os.path.join(data_path, 'val', model_type)

# Set for loading model weights
LOAD_MODEL = False
load_epoch_num = 0
load_model_dir = os.path.join(data_path, 'checkpoint')
load_model_name = os.path.join(data_path, 'checkpoint',
                               f'{model_type}.epoch{load_epoch_num}.pth.tar')
evl_folder_name = os.path.join(data_path, 'evaluation', model_type)

# Load the class weights of all model types
weight_path = os.path.join(data_path, 'class_weight_for_training.txt')
with open(weight_path, "r") as file:
    weight_all = json.load(file)
class_weight = weight_all[model_type]
CLASS_WEIGHTS = torch.FloatTensor(class_weight).to(device=DEVICE)

# Set model channels
in_channels = len(map_dirs)
out_channels = len(class_weight)
print(in_channels, out_channels)


1 5


In [6]:
print(class_weight)

[1, 83, 202, 76, 27]


In [7]:
def train_fn(loader, model, optimizer, loss_fn, scaler):
    loop = tqdm(loader)
    if BATCH_1_TEST:
        data, targets = next(iter(loader))
        for epoch in range(1000):
            data = data.to(device=DEVICE)
            targets = targets.long().to(device=DEVICE)
            # forward
            with torch.cuda.amp.autocast():
                predictions = model(data)
                print(predictions.shape)
                print(targets.shape)
                loss = loss_fn(predictions, targets)
                # print(predictions)
                print(loss)

            # backward
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            # update tqdm loop
            loop.set_postfix(loss=loss.item())

    else:
        for batch_idx, (data, targets) in enumerate(loop):
            data = data.to(device=DEVICE)
            targets = targets.long().to(device=DEVICE)
            # forward
            with torch.cuda.amp.autocast():
                predictions = model(data)
                loss = loss_fn(predictions, targets)
            # backward
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            # update tqdm loop
            loop.set_postfix(loss=loss.item())

In [8]:
# import torch
# from torch.utils.data import DataLoader
# import numpy as np
# import os
# from torch.utils.data import Dataset


# class MapModelDataset(Dataset):

#     def __init__(self, map_dir, model_dir, model_type):
#         self.map_dir = map_dir
#         self.model_dir = model_dir
#         self.maps = os.listdir(map_dir)
#         self.model_type = model_type

#     def __len__(self):
#         return len(self.maps)

#     def __getitem__(self, index):
#         map_path = os.path.join(self.map_dir, self.maps[index])
#         model_path = os.path.join(
#             self.model_dir,
#             self.maps[index].replace('map', "model_" + str(self.model_type)))
#         map = np.expand_dims(np.load(map_path),
#                              axis=0)  # add feature dimension
#         # print(map.shape)
#         model = np.load(model_path)
#         # model = np.expand_dims(np.load(model_path), axis=0)         # add feature dimension
#         # print(model.shape)
#         return map, model


# def get_loaders(
#     model_type,
#     train_map_dir,
#     train_model_dir,
#     val_map_dir,
#     val_model_dir,
#     batch_size,
#     num_workers=4,
#     pin_memory=True,
# ):
#     train_ds = MapModelDataset(map_dir=train_map_dir,
#                                model_dir=train_model_dir,
#                                model_type=model_type)

#     train_loader = DataLoader(
#         train_ds,
#         batch_size=batch_size,
#         num_workers=num_workers,
#         pin_memory=pin_memory,
#         shuffle=True,
#     )

#     val_ds = MapModelDataset(map_dir=val_map_dir,
#                              model_dir=val_model_dir,
#                              model_type=model_type)

#     val_loader = DataLoader(
#         val_ds,
#         batch_size=batch_size,
#         num_workers=num_workers,
#         pin_memory=pin_memory,
#         shuffle=False,
#     )

#     return train_loader, val_loader



# train_loader, val_loader = get_loaders(model_type=model_type,
#                                        train_map_dir=train_map_dirs[0],
#                                        train_model_dir=train_model_dir,
#                                        val_map_dir=val_map_dirs[0],
#                                        val_model_dir=val_model_dir,
#                                        batch_size=batch_size,
#                                        num_workers=num_workers,
#                                        pin_memory=pin_memory)

In [9]:
model = UNET(in_channels=in_channels, out_channels=out_channels).to(DEVICE)
# loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scaler = torch.cuda.amp.GradScaler()

In [10]:

train_loader, val_loader = get_loaders(train_map_dirs=train_map_dirs,
                                        train_model_dir=train_model_dir,
                                        val_map_dirs=val_map_dirs,
                                        val_model_dir=val_model_dir,
                                        batch_size=batch_size,
                                        num_workers=num_workers,
                                        pin_memory=pin_memory)

In [11]:
# train_fn(train_loader, model, optimizer, loss_fn, scaler)

In [19]:
data, targets = next(iter(train_loader))
targets = targets.long()


In [20]:
print(torch.max(targets))
print(torch.min(targets))

tensor(24)
tensor(0)
